In [2]:
# Supporting libraries
import os
from dotenv import load_dotenv
import time
#dotenv
from openai import OpenAI
from PyPDF2 import PdfReader 
import pandas as pd
import os
from pathlib import Path

C:\Users\Buket Fildisi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
openai_api_key=""
os.environ["OPENAI_API_KEY"] = openai_api_key

In [6]:
# Extract full text
import pdfplumber
import nltk
nltk.download("punkt")
pdf_text = ""
with pdfplumber.open("C:/Users/Buket Fildisi/Desktop/PhD/LLM/SDR__Implementation_Update_2024.pdf") as pdf:
    for page in pdf.pages:
        pdf_text += page.extract_text() + "\n"

# Split into sentences
sentences = nltk.sent_tokenize(pdf_text)
sentences

[nltk_data] Downloading package punkt to C:\Users\Buket
[nltk_data]     Fildisi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['Sustainability Disclosure\nRequirements: Implementation\nUpdate 2024\nIntroduction - Sustainability Disclosure Requirements\n1.1 Mobilising Green Investment: 2023 Green Finance Strategy (2023) set out\nthe UK Government’s long-term strategy to ensure investors and consumers can\naccess the information they need to inform capital allocation in line with their\nsustainability preferences or goals.1 Central to this strategy is our plan for economy\nwide Sustainability Disclosure Requirements (SDR), as first set out in Greening\nFinance: A Roadmap to Sustainable Investing (2021).2 Building on global best\npractice and leading standards, SDR is a framework to facilitate and streamline the\nflow of robust, decision useful information between corporates, consumers and\ninvestors and capital markets.',
 '1.2 Financial markets are global, and international interoperability is a key\nprinciple of the UK’s approach to SDR.',
 'The 2023 Green Finance Strategy contained\na commitment to provide a

In [116]:
pdf_text

'Sustainability Disclosure\nRequirements: Implementation\nUpdate 2024\nIntroduction - Sustainability Disclosure Requirements\n1.1 Mobilising Green Investment: 2023 Green Finance Strategy (2023) set out\nthe UK Government’s long-term strategy to ensure investors and consumers can\naccess the information they need to inform capital allocation in line with their\nsustainability preferences or goals.1 Central to this strategy is our plan for economy\nwide Sustainability Disclosure Requirements (SDR), as first set out in Greening\nFinance: A Roadmap to Sustainable Investing (2021).2 Building on global best\npractice and leading standards, SDR is a framework to facilitate and streamline the\nflow of robust, decision useful information between corporates, consumers and\ninvestors and capital markets.\n1.2 Financial markets are global, and international interoperability is a key\nprinciple of the UK’s approach to SDR. The 2023 Green Finance Strategy contained\na commitment to provide an SDR im

In [7]:
sentences[1]

'1.2 Financial markets are global, and international interoperability is a key\nprinciple of the UK’s approach to SDR.'

In [8]:
import string
import re
def text_clean_1(text):
    text = text.lower()
    text= text.replace("\n","");
    text= text.replace("            ","");
    return text

In [10]:
len(sentences)

62

In [11]:
sdr_clean=[]
for i in range(len(sentences)):
    sdr_clean.append(text_clean_1(str(sentences[i])))
#print(len(t_parag))
sdr_clean

['sustainability disclosurerequirements: implementationupdate 2024introduction - sustainability disclosure requirements1.1 mobilising green investment: 2023 green finance strategy (2023) set outthe uk government’s long-term strategy to ensure investors and consumers canaccess the information they need to inform capital allocation in line with theirsustainability preferences or goals.1 central to this strategy is our plan for economywide sustainability disclosure requirements (sdr), as first set out in greeningfinance: a roadmap to sustainable investing (2021).2 building on global bestpractice and leading standards, sdr is a framework to facilitate and streamline theflow of robust, decision useful information between corporates, consumers andinvestors and capital markets.',
 '1.2 financial markets are global, and international interoperability is a keyprinciple of the uk’s approach to sdr.',
 'the 2023 green finance strategy containeda commitment to provide an sdr implementation update 

In [12]:
type(sdr_clean[1])

str

In [18]:
len(sdr_clean)

62

In [24]:
import os
# from openai import OpenAI  <-- Replace this
from langfuse.openai import OpenAI

# 1. Initialize credentials
os.environ["LANGFUSE_PUBLIC_KEY"] = ""
os.environ["LANGFUSE_SECRET_KEY"] = ""
os.environ["LANGFUSE_HOST"] = "" 


In [25]:
client = OpenAI()
# 2. Tell LiteLLM to send data to Langfuse
litellm.success_callback = ["langfuse"]
# 2. Use as normal - Langfuse automatically creates a trace
response = client.chat.completions.create(
    model="gpt-4o-mini",
    name="summary-gen", # Optional: Name the generation in Langfuse
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain sustainable finance concepts in laymens terms."}
    ]
    #,metadata={"customer_id": "user_1234"} # Optional: Add custom metadata
)

print(response.choices[0].message.content)

Sure! Sustainable finance is a way of managing money that focuses on making sure that financial decisions are good for people and the planet, not just for making a profit. Here are some key concepts in simple terms:

1. **Sustainability**: This means keeping our environment healthy and making sure our society works well for everyone, both now and in the future. It’s about balancing economic growth with protecting nature and ensuring social equity.

2. **Responsible Investing**: This is when investors choose to put their money into companies or projects that are doing good things, like using clean energy or treating their workers fairly. They avoid investing in businesses that harm the environment or exploit people.

3. **Environmental, Social, and Governance (ESG) Criteria**: These are the guidelines investors use to evaluate how well a company is doing in terms of sustainability:
   - **Environmental**: How does the company impact the planet? Do they pollute or use renewable resources

In [51]:
prompt_list=[f"Extract related entities from these {sdr_clean[1]}", f"Give me the topic of this {sdr_clean[1]}"]
print(type(prompt_list))
print(len(prompt_list))

<class 'list'>
2


In [47]:
import pandas as pd

def research_assistant(prompt_list):
    all_results = []

    for prompt in prompt_list:
        print(f"Running prompt: {prompt[:50]}...") # Progress tracker
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            # Note: 'name' is usually for the 'message' role in some SDKs, 
            # but for a standard call, we just use messages.
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Extract the content
        content = response.choices[0].message.content
        
        # Store as a dictionary for easy conversion to Excel/DataFrame
        all_results.append({
            "Prompt": prompt,
            "Response": content
        })
        
    return all_results

# Your prompt list
prompt_list = [
    f"Extract entities from these {sdr_clean[1]}", 
    f"Give me the topic of this {sdr_clean[1]}"
]
for i in range(len(sdr_clean)):
    print(i)
    print( f"Extract entities from this sentence {sdr_clean[i]}")

# Execute
results = research_assistant(prompt_list)
results

Running prompt: Extract entities from these 1.2 financial markets ...
Running prompt: Give me the topic of this 1.2 financial markets ar...


[{'Prompt': 'Extract entities from these 1.2 financial markets are global, and international interoperability is a keyprinciple of the uk’s approach to sdr.',
  'Response': 'Here are the extracted entities from the provided text:\n\n1. **Financial markets** - General concept.\n2. **Global** - Descriptor for the financial markets.\n3. **International interoperability** - Concept/principle.\n4. **UK (United Kingdom)** - Country.\n5. **SDR (Special Drawing Rights)** - Financial instrument or concept. \n\nWould you like more detailed explanations or context for any of these entities?'},
 {'Prompt': 'Give me the topic of this 1.2 financial markets are global, and international interoperability is a keyprinciple of the uk’s approach to sdr.',
  'Response': "The topic of the statement refers to the global nature of financial markets and the importance of international interoperability, particularly in the context of the UK's approach to Special Drawing Rights (SDR). This highlights how countr

In [127]:
from langfuse import observe
import os
client = OpenAI()

@observe()
def research_assistant(text: str):
    # Step 1: Search/Reasoning (Generation 1)
    entities = client.chat.completions.create(
        model="gpt-4o-mini",
        name="planning-step",
        messages=[{"role": "user", "content": f"""You are aiming to create a knowledge graph. Your task is to extract all defined terms, entities, relationships between entities and legally relevant concepts from the following Sustainable Disclosure Requirements (SDR) document {text}.

Instructions:
- Extract every defined term or entity-relationships-entity as a clean list.
- Categorise the entities as concept, organisations, sectors, group, financial ters or metrics
- Include names of organizations, roles, regulations, laws, and standards mentioned.
- Do not deduplicate similar terms—list each occurrence separately if it appears in the text.
- For each extracted term, include:
- The exact term or entity as it appears
- If available, its definition or a short excerpt of the sentence where it was defined
- Present the results in a clean bullet-point list in markdown."""}]
        ).choices[0].message.content   
    return entities   

    # Step 2: Final Answer (Generation 2)
   # final_report = client.chat.completions.create(
   #     model="gpt-4o-mini",
   #     name="final-report",
   #     messages=[{"role": "user", "content": f"Write a report based on this plan: {plan}"}]
   # ).choices[0].message.content
   #   

# This call will show up in Langfuse as one 'Trace' containing two 'Generations'
result=research_assistant(pdf_text)

In [201]:
import pandas as pd
import json

def run_research_and_export(sentence_list, manual_prompt, file_name="SDR_Output_P1.xlsx"):
    flattened_data = []
    print(manual_prompt)
    for s_index, sentence in enumerate(sentence_list):
        print(f"Processing {s_index + 1}/{len(sentence_list)}...")
        print(sentence)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": f"{manual_prompt}\n\nSentence: {sentence}"}]
        )
        content = response.choices[0].message.content.strip()
        print(content)

        try:
            # Clean and parse JSON
            clean_content = content.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean_content)
            
            # Navigate to the list of terms
            # Based on your image, it looks like 'terms' is the main key
            items = data.get('terms', data.get('entities', []))

            if isinstance(items, list):
                for item in items:
                    if isinstance(item, dict):
                        # MAPPING: 'term' -> Entity, 'category' -> Category
                        flattened_data.append({
                            'Sentence': sentence.replace('\n', ' '),
                            'Category': item.get('category', 'N/A'),
                            'Entity': item.get('term', 'N/A'),
                            'Value': item.get('term', 'N/A') # Or use a description if available
                        })
            elif isinstance(items, dict):
                # Fallback for the { 'UK': 'United Kingdom' } format
                for cat, entities in items.items():
                    if isinstance(entities, list):
                        for e in entities:
                            flattened_data.append({
                                'Sentence': sentence.replace('\n', ' '),
                                'Category': cat,
                                'Entity': e,
                                'Value': e
                            })

        except Exception as e:
            print(f"Error parsing JSON: {e}")
            flattened_data.append({
                'Sentence': sentence,
                'Category': 'Error',
                'Entity': 'Raw Response',
                'Value': content
            })

    # Export
    df = pd.DataFrame(flattened_data)
    df.to_excel(file_name, index=False)
    
    print(f"✅ Fixed! Data exported to {file_name}")

In [204]:
prompt1="""Extract all defined terms, ENTITIES and their categories from this SUSTAINABLE DISCLOSURE REQUIREMENTS document AS A name entity recognizer specialist. Output should be in JSON format, Include capitalized terms, terms in quotation marks, and those introduced by phrases like 'means' or 'shall be understood as and only show extracted entities in the output"""

In [211]:
prompt2 = """
You are an expert Ontology Engineer. Extract triples (Subject-Predicate-Object) from the provided regulatory text.
For every entity extracted as a Subject or Object, you MUST also identify its Category.

For each extraction, output exactly in this JSON format:
{
  "triples": [
    {
      "subject": "<entity name>",
      "subject_category": "<category of subject>",
      "predicate": "<relationship verb or phrase>",
      "object": "<entity name>",
      "object_category": "<category of object>"
    }
  ]
}

Guidelines:
- Categorize entities as (e.g., Organization, Concept, Document, Process, Date, or Regulation).
- Only extract explicit relationships defined in the text.
- Provide ONLY the JSON array. Do not include introductory text.
"""

In [215]:
def run_research_and_export_rel_v2(sentence_list, manual_prompt, file_name="SDR_P2.xlsx"):
    flattened_data = []
    
    for s_index, sentence in enumerate(sentence_list):
        print(f"Processing {s_index + 1}/{len(sentence_list)}...")
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": f"{manual_prompt}\n\nSentence: {sentence}"}],
            temperature=0 # Set to 0 for consistent extraction
        )
        
        content = response.choices[0].message.content.strip()

        try:
            # Clean and parse JSON
            clean_content = content.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean_content)
            
            # Access the 'triples' key defined in the prompt
            items = data.get('triples', [])

            for item in items:
                flattened_data.append({
                    'Sentence': sentence.replace('\n', ' '),
                    'Subject': item.get('subject', 'N/A'),
                    'Subject_Category': item.get('subject_category', 'N/A'),
                    'Predicate': item.get('predicate', 'N/A'),
                    'Object': item.get('object', 'N/A'),
                    'Object_Category': item.get('object_category', 'N/A')
                })

        except Exception as e:
            print(f"Error parsing JSON on sentence {s_index}: {e}")
            # Optional: Log the error to the dataframe to see what failed
            
    # Export to Excel
    df = pd.DataFrame(flattened_data)
    df.to_excel(file_name, index=False)
    print(f"✅ Success! Triples with categories exported to {file_name}")

# To run:
# run_research_and_export_rel_v2(sentence_list, prompt2)

In [216]:
run_research_and_export(sdr_clean, prompt1)
run_research_and_export_rel_v2(sdr_clean, prompt2)

Processing 1/62...
Processing 2/62...
Processing 3/62...
Processing 4/62...
Processing 5/62...
Processing 6/62...
Processing 7/62...
Processing 8/62...
Processing 9/62...
Processing 10/62...
Processing 11/62...
Processing 12/62...
Processing 13/62...
Processing 14/62...
Processing 15/62...
Processing 16/62...
Processing 17/62...
Processing 18/62...
Processing 19/62...
Processing 20/62...
Processing 21/62...
Processing 22/62...
Processing 23/62...
Processing 24/62...
Processing 25/62...
Processing 26/62...
Processing 27/62...
Processing 28/62...
Processing 29/62...
Processing 30/62...
Processing 31/62...
Processing 32/62...
Processing 33/62...
Processing 34/62...
Processing 35/62...
Processing 36/62...
Processing 37/62...
Processing 38/62...
Processing 39/62...
Processing 40/62...
Processing 41/62...
Processing 42/62...
Processing 43/62...
Processing 44/62...
Processing 45/62...
Processing 46/62...
Processing 47/62...
Processing 48/62...
Processing 49/62...
Processing 50/62...
Processin

In [217]:
data_p1=pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\PhD\LLM\SDR_Output_P1.xlsx")
data_p2= pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\PhD\LLM\SDR_P2.xlsx")
data_neo4j= pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\PhD\LLM\SDR_NEO4J_ENTITYLabels.xlsx")
gold= pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\SDR_ValidationSet.xlsx")
gold.head()

,Source Sentence,Entity,Category
0,1.11 Given the overlap between IFRS S2 and the...,IFRS S2,STANDARD
1,1.11 Given the overlap between IFRS S2 and the...,TPT Disclosure Framework,FRAMEWORK
2,1.11 Given the overlap between IFRS S2 and the...,FCA,Organization
3,1.11 Given the overlap between IFRS S2 and the...,UK-endorsed ISSB standards,STANDARD
4,1.11 Given the overlap between IFRS S2 and the...,ISSB,Organization


In [261]:
data_neo4j_trip= pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\PhD\LLM\SDR_NEO4J_ENTITYLabels.xlsx", sheet_name="Triples")
data_neo4j_trip.head()

,Subject,Predicate,Object
0,Government,AIMS_TO_MAKE_AVAILABLE,UK-endorsed ISSB standards
1,Government,ANNOUNCED_INTENTION,consult on whether to broaden the scope of SDR...
2,UK Sustainability Reporting Standards,APPLIES_TO,UK companies that do not fall within the FCAâ€...
3,investment labels regime for funds based in th...,APPLIES_TO,funds based in the UK
4,anti-greenwashing rule,APPLIES_TO,FCA-authorised firms


In [243]:
gold_triples= pd.read_excel(r"C:\Users\Buket Fildisi\Desktop\SDR_ValidationSet.xlsx", sheet_name="Relationships")
gold_triples

,Sentences,Subject,Subject Category,Predicate,Object,Object Category
0,1.11,IFRS S2,STANDARD,overlaps with,TPT Disclosure Framework,FRAMEWORK
1,1.11,FCA,Organization,plans to consult on,Transition plan disclosures,Concept
2,1.11,FCA,Organization,references,TPT Disclosure Framework,POLICY_MECHANISM
3,1.12,UK Government,Organization,will consult on,Transition plans,Plan
4,1.12,Transition plans,Plan,meets commitment of,Green Finance Strategy,Policy_Strategy
5,1.13,FCA,Organization,finalised,SDR,Regulation
6,1.13,FCA,Organization,finalised,Investment labels regime,Policy_Framework
7,1.13,SDR Policy Statement,Policy_Document,released on,2023-11-28 00:00:00,Date
8,1.13,FCA,Organization,engaged with,Disclosures & Labels Advisory Group,Organization
9,1.14,SDR Package,Regulation,includes,Investment labels,POLICY_MECHANISM


In [244]:
data_p2.head()

,Sentence,Subject,Subject_Category,Predicate,Object,Object_Category
0,sustainability disclosurerequirements: impleme...,UK Government,Organization,set out,2023 Green Finance Strategy,Document
1,sustainability disclosurerequirements: impleme...,2023 Green Finance Strategy,Document,ensure,investors and consumers,Organization
2,sustainability disclosurerequirements: impleme...,investors and consumers,Organization,can access,information,Concept
3,sustainability disclosurerequirements: impleme...,information,Concept,to inform,capital allocation,Process
4,sustainability disclosurerequirements: impleme...,sustainability disclosure requirements (SDR),Concept,is,a framework,Concept


In [224]:
def normalize_text(x):
    return str(x).strip().lower().replace("\n", " ")
alias_map = {
    "sustainability disclosure requirements": "sdr",
    "financial conduct authority": "fca"
}

In [225]:
gold[['Category', 'Entity']]

,Category,Entity
0,STANDARD,IFRS S2
1,FRAMEWORK,TPT Disclosure Framework
2,Organization,FCA
3,STANDARD,UK-endorsed ISSB standards
4,Organization,ISSB
5,POLICY_MECHANISM,transition plan disclosures
6,Organization,Government
7,Country,UK
8,FINANCIAL_ACTOR,UK’s largest companies
9,Plan,transition planning


In [237]:
data_p2.head()

,Sentence,Subject,Subject_Category,Predicate,Object,Object_Category
0,sustainability disclosurerequirements: impleme...,UK Government,Organization,set out,2023 Green Finance Strategy,Document
1,sustainability disclosurerequirements: impleme...,2023 Green Finance Strategy,Document,ensure,investors and consumers,Organization
2,sustainability disclosurerequirements: impleme...,investors and consumers,Organization,can access,information,Concept
3,sustainability disclosurerequirements: impleme...,information,Concept,to inform,capital allocation,Process
4,sustainability disclosurerequirements: impleme...,sustainability disclosure requirements (SDR),Concept,is,a framework,Concept


In [257]:
data_neo4j.head()

,Entity,Category
0,2023-10-01 00:00:00,Date
1,make rules for overseas funds,Action
2,Final anti-greenwashing guidance,"Policy, Action"
3,UK Green Taxonomy,Concept
4,consultation,"Report, Action"


In [258]:
# Convert specific columns to a list of dictionaries
# This replaces your old 'item in extracted_data' logic
entity_data = data_p1[['Category', 'Entity', 'Sentence']].to_dict('records')
entity_data_nj = data_neo4j[['Category', 'Entity']].to_dict('records')
print(entity_data_nj)
# If you just need a list of strings for the embedding model input:
entities_list = entity_data_nj['Entity'].astype(str).tolist()

[{'Category': ' Date', 'Entity': datetime.datetime(2023, 10, 1, 0, 0)}, {'Category': ' Action', 'Entity': 'make rules for overseas funds'}, {'Category': ' Policy, Action', 'Entity': 'Final anti-greenwashing guidance'}, {'Category': ' Concept', 'Entity': 'UK Green Taxonomy'}, {'Category': ' Report, Action', 'Entity': 'consultation'}, {'Category': ' Organization', 'Entity': 'Financial Conduct Authority (FCA)'}, {'Category': ' Organization, Government Entity', 'Entity': 'UK Government'}, {'Category': ' Action', 'Entity': 'Consultation on extending the regime to portfolio managers and final rules'}, {'Category': ' Action', 'Entity': 'Consultation on strengthening transition plan disclosure expectations in line with TPT framework'}, {'Category': ' Action', 'Entity': 'Staggered implementation of final rules'}, {'Category': ' Concept,  Policy', 'Entity': 'UK Sustainability Reporting Standards'}, {'Category': ' Action', 'Entity': 'Assessment concludes and recommendations provided'}, {'Category

TypeError: list indices must be integers or slices, not str

In [260]:
import pandas as pd
from scipy.spatial.distance import cosine

# 1. Prepare the Semantic Identity for Gold and AI
# We combine Entity and Category into a single concept string
gold['semantic_identity'] = (    "Entity: " + gold['Entity'].astype(str) +     " | Category: " + gold['Category'].astype(str))

data_p1['semantic_identity'] = (    "Entity: " + data_p1['Entity'].astype(str) +     " | Category: " + data_p1['Category'].astype(str))
data_neo4j['semantic_identity'] = (    "Entity: " + data_neo4j['Entity'].astype(str) +     " | Category: " + data_neo4j['Category'].astype(str))
# 2. Get Embeddings for both lists (Bulk call is more efficient)
gold_identities = gold['semantic_identity'].tolist()
ai_identities = data_neo4j['semantic_identity'].tolist()

# Get embeddings from your 'emb' model
gold_vecs = client.embeddings.create(model="text-embedding-3-small", input=gold_identities).data
ai_vecs = client.embeddings.create(model="text-embedding-3-small", input=ai_identities).data

# Extract actual vectors
gold_vectors = [v.embedding for v in gold_vecs]
ai_vectors = [v.embedding for v in ai_vecs]

# 3. Calculate "Semantic Proximity" for each AI result
def get_validation_score(ai_vec, gold_vectors):
    # Distance to the nearest human-verified concept
    distances = [cosine(ai_vec, g_vec) for g_vec in gold_vectors]
    return min(distances)

# Apply to your 246 extractions
data_neo4j['validation_distance'] = [get_validation_score(vec, gold_vectors) for vec in ai_vectors]

# 4. Resulting DataFrame (includes entity, sentence, and category)
# Higher distance = lower confidence in the Entity+Category pair
final_validation = data_neo4j[['Entity', 'Category', 'validation_distance']]
final_validation.to_excel("SDR_Full_Validation_n4j.xlsx", index=False)

In [231]:
gold.head()

,Source Sentence,Entity,Category,semantic_identity
0,1.11 Given the overlap between IFRS S2 and the...,IFRS S2,STANDARD,Entity: IFRS S2 | Category: STANDARD
1,1.11 Given the overlap between IFRS S2 and the...,TPT Disclosure Framework,FRAMEWORK,Entity: TPT Disclosure Framework | Category: F...
2,1.11 Given the overlap between IFRS S2 and the...,FCA,Organization,Entity: FCA | Category: Organization
3,1.11 Given the overlap between IFRS S2 and the...,UK-endorsed ISSB standards,STANDARD,Entity: UK-endorsed ISSB standards | Category:...
4,1.11 Given the overlap between IFRS S2 and the...,ISSB,Organization,Entity: ISSB | Category: Organization


In [232]:
judge_template = """
You are an expert in SDR (Sustainable Disclosure Requirements Document).

Entity: "{entity}"
Assigned Label: "{label}"

Task:
- Decide if the label correctly describes the entity.
- Return JSON with fields:
  {{
    "is_correct": true or false,
    "reasoning": "short explanation"
  }}
"""

In [233]:
def llm_judge(prompt, client):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    return response.choices[0].message.content

In [274]:
import json
import pandas as pd

def call_llm_judge_on_results(df, gold_anchors, client):
    # 1. Create a list of the 43 manual labels to give the LLM context
    anchor_summary = gold_anchors[['Entity', 'Category']].to_dict('records')
    
    results = []

    for index, row in df.iterrows():
        # Construct the specific prompt for this row
        prompt = f"""
        You are an expert Ontology Auditor for UK Sustainability Disclosure Requirements (SDR).
        
        EXTRACTED ENTITY: "{row['Entity']}"
        PREDICTED CATEGORY: "{row['Category']}"
        
        VALIDATION ANCHORS (Reference Examples): {anchor_summary}
        
        TASK:
        1. Compare the extracted pair to the gold anchors.
        2. Is this a logically valid SDR entity and correctly categorized based on the context?
        
        RETURN JSON ONLY:
        {{
            "is_valid": true,
            "confidence": 0.95,
            "reasoning": "Explanation here"
        }}
        """
        
        # Call your existing llm_judge function
        raw_response = llm_judge(prompt, client)
        
        try:
            # Clean the response in case the LLM added markdown code blocks
            clean_json = raw_response.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean_json)
            results.append(parsed)
        except Exception as e:
            results.append({"is_valid": None, "confidence": 0, "reasoning": f"Error: {str(e)}"})

    # 2. Add the results back to your DataFrame
    res_df = pd.DataFrame(results)
    final_df = pd.concat([df.reset_index(drop=True), res_df], axis=1)
    
    return final_df

# To execute:
# final_validated_df = call_llm_judge_on_results(data, gold, client)

In [272]:
gold.head()

,Source Sentence,Entity,Category,semantic_identity
0,1.11 Given the overlap between IFRS S2 and the...,IFRS S2,STANDARD,Entity: IFRS S2 | Category: STANDARD
1,1.11 Given the overlap between IFRS S2 and the...,TPT Disclosure Framework,FRAMEWORK,Entity: TPT Disclosure Framework | Category: F...
2,1.11 Given the overlap between IFRS S2 and the...,FCA,Organization,Entity: FCA | Category: Organization
3,1.11 Given the overlap between IFRS S2 and the...,UK-endorsed ISSB standards,STANDARD,Entity: UK-endorsed ISSB standards | Category:...
4,1.11 Given the overlap between IFRS S2 and the...,ISSB,Organization,Entity: ISSB | Category: Organization


In [275]:
final_validated_n4j = call_llm_judge_on_results(data_neo4j, gold, client)

In [276]:
final_validated_n4j.to_excel("SDR_LLMasajudge_Results_n4j.xlsx")

In [236]:
final_validated_df.to_excel("SDR_LLMasajudge_Results_P1.xlsx")

In [249]:
data_p2.columns

Index(['Sentence', 'Subject', 'Subject_Category', 'Predicate', 'Object',
       'Object_Category'],
      dtype='object')

In [250]:
###for triples
import pandas as pd
from scipy.spatial.distance import cosine

# 1. Create the Semantic Identity for the Triples
# Format: "Subject (Category) -> Predicate -> Object (Category)"
gold_triples['triple_identity'] = (
    "Subject: " + gold_triples['Subject'].astype(str) + " (" + gold_triples['Subject Category'].astype(str) + ") " +
    "| Predicate: " + gold_triples['Predicate'].astype(str) + 
    " | Object: " + gold_triples['Object'].astype(str) + " (" + gold_triples['Object Category'].astype(str) + ")"
)

data_p2['triple_identity'] = (
    "Subject: " + data_p2['Subject'].astype(str) + " (" + data_p2['Subject_Category'].astype(str) + ") " +
    "| Predicate: " + data_p2['Predicate'].astype(str) + 
    " | Object: " + data_p2['Object'].astype(str) + " (" + data_p2['Object_Category'].astype(str) + ")"
)

In [262]:
gold_triples['triple_identity'] = (
    "Subject: " + gold_triples['Subject'].astype(str) + "| Predicate: " + gold_triples['Predicate'].astype(str) +   " | Object: " + gold_triples['Object'].astype(str) 
)

In [263]:
data_neo4j_trip['triple_identity'] = (
    "Subject: " + data_neo4j_trip['Subject'].astype(str) +     "| Predicate: " + data_neo4j_trip['Predicate'].astype(str) + 
    " | Object: " + data_neo4j_trip['Object'].astype(str) )

In [264]:
# 2. Get Embeddings for Triple Identities
gold_triple_strings = gold_triples['triple_identity'].tolist()
ai_triple_strings = data_neo4j_trip['triple_identity'].tolist()

# Get embeddings
gold_triple_res = client.embeddings.create(model="text-embedding-3-small", input=gold_triple_strings).data
ai_triple_res = client.embeddings.create(model="text-embedding-3-small", input=ai_triple_strings).data

# Extract vectors
gold_triple_vectors = [v.embedding for v in gold_triple_res]
ai_triple_vectors = [v.embedding for v in ai_triple_res]

In [265]:
# 3. Calculate "Topological Proximity"
def get_triple_validation_score(ai_vec, gold_vectors):
    # Find the distance to the closest manual triple
    distances = [cosine(ai_vec, g_vec) for g_vec in gold_vectors]
    return min(distances)

# Apply to your Neo4j extractions
data_neo4j_trip['triple_validation_distance'] = [get_triple_validation_score(vec, gold_triple_vectors) for vec in ai_triple_vectors]

# 4. Export Final Evaluation
#final_triple_validation = data_p2[['Sentence', 'Subject', 'Predicate', 'Object', 'triple_validation_distance']]
final_triple_validation = data_neo4j_trip[['Subject', 'Predicate', 'Object', 'triple_validation_distance']]
final_triple_validation.to_excel("SDR_Triple_Validation_neo4jTrip.xlsx", index=False)

In [267]:
data_neo4j_trip.head()

,Subject,Predicate,Object,triple_identity,triple_validation_distance
0,Government,AIMS_TO_MAKE_AVAILABLE,UK-endorsed ISSB standards,Subject: Government| Predicate: AIMS_TO_MAKE_A...,0.494870
1,Government,ANNOUNCED_INTENTION,consult on whether to broaden the scope of SDR...,Subject: Government| Predicate: ANNOUNCED_INTE...,0.220101
2,UK Sustainability Reporting Standards,APPLIES_TO,UK companies that do not fall within the FCAâ€...,Subject: UK Sustainability Reporting Standards...,0.443176
3,investment labels regime for funds based in th...,APPLIES_TO,funds based in the UK,Subject: investment labels regime for funds ba...,0.244124
4,anti-greenwashing rule,APPLIES_TO,FCA-authorised firms,Subject: anti-greenwashing rule| Predicate: AP...,0.248306


In [268]:
##llm-as-a-judge for triples
import json
import pandas as pd

def call_llm_judge_on_triples(df_ai_triples, df_gold_triples, client):
    # 1. Prepare the Relationship Gold Set for the LLM context
    # We provide the Judge with your 17 human-verified triples as reference
    anchor_summary = df_gold_triples[['Subject', 'Predicate', 'Object']].to_dict('records')
    
    results = []

    for index, row in df_ai_triples.iterrows():
        # Construct the specific prompt for Triple Validation
        prompt = f"""
        You are an expert Knowledge Graph Auditor for the UK Sustainability Disclosure Requirements (SDR).     
        AI-EXTRACTED TRIPLE:
        - Subject: "{row['Subject']}" 
        - Predicate: "{row['Predicate']}"
        - Object: "{row['Object']}" 
        
        GOLD STANDARD EXAMPLES (For Reference): {anchor_summary}
        
        TASK:
        1. Evaluate if the AI-extracted triple accurately represents the relationship described in the CONTEXT SENTENCE.
        2. Verify if the Categories (Subject_Category and Object_Category) are ontologically correct.
        3. Even if the Predicate is not identical to the Gold Standard, is it semantically equivalent and factually true?
        
        RETURN JSON ONLY:
        {{
            "is_valid": true/false,
            "confidence": 0.0-1.0,
            "reasoning": "Brief explanation of why this relationship is valid or a hallucination."
        }}
        """
        
        # Call your existing llm_judge function
        raw_response = llm_judge(prompt, client)
        
        try:
            # Clean and parse JSON
            clean_json = raw_response.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(clean_json)
            results.append(parsed)
        except Exception as e:
            results.append({"is_valid": None, "confidence": 0, "reasoning": f"Error: {str(e)}"})

    # 2. Merge results with the original triple DataFrame
    res_df = pd.DataFrame(results)
    final_df = pd.concat([df_ai_triples.reset_index(drop=True), res_df], axis=1)
    
    return final_df

In [ ]:
# this one has sentences I don't include them for neo4j this belos query is valid for P2
##llm-as-a-judge for triples
#def call_llm_judge_on_triples(df_ai_triples, df_gold_triples, client):
#    # 1. Prepare the Relationship Gold Set for the LLM context
#    # We provide the Judge with your 17 human-verified triples as reference
#    anchor_summary = df_gold_triples[['Subject', 'Predicate', 'Object']].to_dict('records')
#    
#    results = []
#
#    for index, row in df_ai_triples.iterrows():
#        # Construct the specific prompt for Triple Validation
#        prompt = f"""
#        You are an expert Knowledge Graph Auditor for the UK Sustainability Disclosure Requirements (SDR).
#        
#        CONTEXT SENTENCE: "{row['Sentence']}"
#        
#        AI-EXTRACTED TRIPLE:
#        - Subject: "{row['Subject']}" ({row['Subject_Category']})
#        - Predicate: "{row['Predicate']}"
#        - Object: "{row['Object']}" ({row['Object_Category']})
#        
#        GOLD STANDARD EXAMPLES (For Reference): {anchor_summary}
#        
#        TASK:
#        1. Evaluate if the AI-extracted triple accurately represents the relationship described in the CONTEXT SENTENCE.
#        2. Verify if the Categories (Subject_Category and Object_Category) are ontologically correct.
#        3. Even if the Predicate is not identical to the Gold Standard, is it semantically equivalent and factually true?
#        
#        RETURN JSON ONLY:
#        {{
#            "is_valid": true/false,
#            "confidence": 0.0-1.0,
#            "reasoning": "Brief explanation of why this relationship is valid or a hallucination."
#        }}
#        """
#        
#        # Call your existing llm_judge function
#        raw_response = llm_judge(prompt, client)
#        
#        try:
#            # Clean and parse JSON
#            clean_json = raw_response.replace("```json", "").replace("```", "").strip()
#            parsed = json.loads(clean_json)
#            results.append(parsed)
#        except Exception as e:
#            results.append({"is_valid": None, "confidence": 0, "reasoning": f"Error: {str(e)}"})
#
#    # 2. Merge results with the original triple DataFrame
#    res_df = pd.DataFrame(results)
#    final_df = pd.concat([df_ai_triples.reset_index(drop=True), res_df], axis=1)
#    
#    return final_df

In [254]:
final_triples_audit = call_llm_judge_on_triples(data_p2, gold_triples, client)
final_triples_audit

,Sentence,Subject,Subject_Category,Predicate,Object,Object_Category,triple_identity,triple_validation_distance,is_valid,confidence,reasoning
0,sustainability disclosurerequirements: impleme...,UK Government,Organization,set out,2023 Green Finance Strategy,Document,Subject: UK Government (Organization) | Predic...,0.344288,True,0.9,The AI-extracted triple accurately captures th...
1,sustainability disclosurerequirements: impleme...,2023 Green Finance Strategy,Document,ensure,investors and consumers,Organization,Subject: 2023 Green Finance Strategy (Document...,0.330054,False,0.6,The AI-extracted triple inaccurately represent...
2,sustainability disclosurerequirements: impleme...,investors and consumers,Organization,can access,information,Concept,Subject: investors and consumers (Organization...,0.536332,False,0.7,The AI-extracted triple inaccurately identifie...
3,sustainability disclosurerequirements: impleme...,information,Concept,to inform,capital allocation,Process,Subject: information (Concept) | Predicate: to...,0.470465,False,0.7,The AI-extracted triple does not accurately re...
4,sustainability disclosurerequirements: impleme...,sustainability disclosure requirements (SDR),Concept,is,a framework,Concept,Subject: sustainability disclosure requirement...,0.399153,True,0.9,The AI-extracted triple accurately captures th...
...,...,...,...,...,...,...,...,...,...,...,...
205,"to view this licence, visitnationalarchives.go...",this licence,Document,view,nationalarchives.gov.uk/doc/open-government-li...,Document,Subject: this licence (Document) | Predicate: ...,0.541277,False,0.4,The AI-extracted triple inaccurately represent...
206,where we have identified any third party copyr...,third party copyright information,Concept,identified,copyright holders,Organization,Subject: third party copyright information (Co...,0.534748,False,0.4,The AI-extracted triple does not accurately re...
207,where we have identified any third party copyr...,you,Organization,need to obtain permission from,copyright holders,Organization,Subject: you (Organization) | Predicate: need ...,0.535200,False,0.6,The AI-extracted triple misrepresents the rela...
208,this publication is available at: www.gov.uk/o...,this publication,Document,is available at,www.gov.uk/official-documents,Concept,Subject: this publication (Document) | Predica...,0.537520,True,0.9,The AI-extracted triple accurately represents ...


In [255]:
final_triples_audit.to_excel("SDR_LLMasajudge_p2.xlsx")

In [269]:
final_triples_audit_n4jt = call_llm_judge_on_triples(data_neo4j_trip, gold_triples, client)
final_triples_audit_n4jt

,Subject,Predicate,Object,triple_identity,triple_validation_distance,is_valid,confidence,reasoning
0,Government,AIMS_TO_MAKE_AVAILABLE,UK-endorsed ISSB standards,Subject: Government| Predicate: AIMS_TO_MAKE_A...,0.494870,True,0.9,The AI-extracted triple accurately represents ...
1,Government,ANNOUNCED_INTENTION,consult on whether to broaden the scope of SDR...,Subject: Government| Predicate: ANNOUNCED_INTE...,0.220101,True,0.9,The AI-extracted triple accurately represents ...
2,UK Sustainability Reporting Standards,APPLIES_TO,UK companies that do not fall within the FCAâ€...,Subject: UK Sustainability Reporting Standards...,0.443176,True,0.9,The AI-extracted triple accurately represents ...
3,investment labels regime for funds based in th...,APPLIES_TO,funds based in the UK,Subject: investment labels regime for funds ba...,0.244124,True,0.9,The AI-extracted triple accurately represents ...
4,anti-greenwashing rule,APPLIES_TO,FCA-authorised firms,Subject: anti-greenwashing rule| Predicate: AP...,0.248306,True,0.9,The AI-extracted triple accurately represents ...
...,...,...,...,...,...,...,...,...
116,Government,WELCOMES,Taskforce on Nature-related Financial,Subject: Government| Predicate: WELCOMES | Obj...,0.470758,True,0.9,The AI-extracted triple accurately represents ...
117,Government,WELCOMES_COMMITMENT,ISSB,Subject: Government| Predicate: WELCOMES_COMMI...,0.557051,True,0.9,The AI-extracted triple accurately represents ...
118,UK Government,WILL_CONSULT_ON,disclosure requirements against the UK Sustain...,Subject: UK Government| Predicate: WILL_CONSUL...,0.335150,True,0.9,The AI-extracted triple accurately represents ...
119,Government,WILL_CONSULT_ON,how the UKâ€™s largest companies can disclose ...,Subject: Government| Predicate: WILL_CONSULT_O...,0.223054,True,0.9,The AI-extracted triple accurately reflects th...


In [270]:
final_triples_audit_n4jt.to_excel("SDR_LLMasajudge_neo4jTrip.xlsx")

In [171]:
import pandas as pd

# Step A: Get the list of entity strings for the embedding model
entities_list = df['Entity'].astype(str).tolist()

# Step B: Get Embeddings (using the same 'emb' logic from your notebook)
# This creates vectors for all 246 (or more) entities at once
embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=entities_list
)

# Step C: Combine everything back into a DataFrame
# We keep 'entity' and 'sentence' as required for your Knowledge Graph
records = []
for i, row in enumerate(df.to_dict('records')):
    records.append({
        'entity': row['Entity'],
        'category': row['Category'],
        'sentence': row['Sentence'],
        'embedding': embedding_response.data[i].embedding
    })

df_vectorized = pd.DataFrame(records)

In [172]:
df_vectorized

,entity,category,sentence,embedding
0,Sustainability Disclosure Requirements,N/A,sustainability disclosurerequirements: impleme...,"[0.005727109964936972, 0.020185058936476707, 0..."
1,Implementation Update 2024,N/A,sustainability disclosurerequirements: impleme...,"[-0.016069985926151276, -0.04543142393231392, ..."
2,2023 Green Finance Strategy,N/A,sustainability disclosurerequirements: impleme...,"[-0.007716048043221235, 0.010138679295778275, ..."
3,UK Government,N/A,sustainability disclosurerequirements: impleme...,"[-0.010704738087952137, 0.01040758565068245, 0..."
4,Green Investment,N/A,sustainability disclosurerequirements: impleme...,"[-0.007596506271511316, 0.03976994380354881, 0..."
...,...,...,...,...
147,N/A,N/A,where we have identified any third party copyr...,"[-0.00831184908747673, -0.0018222130602225661,..."
148,N/A,N/A,where we have identified any third party copyr...,"[-0.00831184908747673, -0.0018222130602225661,..."
149,N/A,N/A,this publication is available at: www.gov.uk/o...,"[-0.00831184908747673, -0.0018222130602225661,..."
150,N/A,N/A,this publication is available at: www.gov.uk/o...,"[-0.00831184908747673, -0.0018222130602225661,..."
